In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from openai import OpenAI
openai_client = OpenAI()

from ingest import *

In [6]:
documents = load_faq_data()
index = build_index(documents)

In [3]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I still join?"},]

def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages
    )
    return response.output_text

In [9]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

call = response.output

In [14]:
print(call)

[ResponseFunctionToolCall(arguments='{"query":"Can I still join if I just discovered the course? enrollment late join join now after course started"}', call_id='call_MnlaMhogL2x8Zn3h2IeQAlTe', name='search', type='function_call', id='fc_0dd5f7845d4af422006a39012dba20819faf0b25dde2b37e8b', namespace=None, status='completed')]


In [15]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [16]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "9f689c185f",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I missed the first homework - can I still get a certificate?",
    "answer": "Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certifi

In [17]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [18]:
messages

[{'role': 'user',
  'content': 'I just discovered the course. Can I still join?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I still join if I just discovered the course? enrollment late join join now after course started"}', call_id='call_MnlaMhogL2x8Zn3h2IeQAlTe', name='search', type='function_call', id='fc_0dd5f7845d4af422006a39012dba20819faf0b25dde2b37e8b', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_MnlaMhogL2x8Zn3h2IeQAlTe',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "9f689c185f",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I missed the first homework - can 

In [19]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [20]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(820, 29)

In [21]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176
